In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1. Load the translated data

In [2]:
INPUT_FILE = "/content/drive/MyDrive/PSA_Translated_NLLB.csv"

!pip install pandas regex -q --break-system-packages
import pandas as pd

df_clean = pd.read_csv(INPUT_FILE)
print("Shape:", df_clean.shape)
print("Missing Dholuo:", df_clean['Dholuo'].isna().sum())
df_clean.head()

Shape: (9539, 5)
Missing Dholuo: 0


,PSA_ID,Domain,English,Kiswahili,Dholuo
0,PSA000001,Health,"PRESS RELEASE: JUNE 29, 2020 The EU through it...","TAARIFA KWA VYOMBO VYA HABARI: JUNI 29, 2020 E...","WECHE MA MUMA: Jun 29, 2020 EU kokalo kuom mig..."
1,PSA000002,Health,This grant will be used by the WHO to support ...,Ruzuku hii itatumiwa na WHO kuunga mkono juhud...,Riwruok mar WHO biro tiyo gi konyno e siro kin...
2,PSA000003,Health,"Specifically, WHO Kenya will boost the respons...","Hasa, WHO Kenya itaongeza juhudi za kukabilian...","E yo makende, WHO Kenya biro jiwo kinda mar dw..."
3,PSA000004,Health,Strengthening clinical care for high-consequen...,Kuimarisha huduma ya kimatibabu kwa magonjwa y...,Tego chenro mar thieth mar thieth mar tuoche m...
4,PSA000005,Health,"In recent years, investments in surveillance, ...","Katika miaka ya hivi karibuni, uwekezaji katik...","E higini mosekalo, gweth ma osetigo e rito ji,..."


2. Fix PSA_IDs and flag template gaps

In [3]:
# Rows like "comments on for community development" — a template variable was deleted
template_gap_pattern = r'\b(?:on|regarding|about)\s+for\b|\b(?:on|regarding)\s+to\b'
df_clean['has_template_gap'] = df_clean['English'].astype(str).str.contains(template_gap_pattern, regex=True, na=False)
print("Flagged for review:", df_clean['has_template_gap'].sum())

Flagged for review: 14


In [4]:
# Re-key any colliding PSA_IDs (no-op if the source file's IDs are already unique, as PSA_Translated_NLLB.csv's are)
dup_mask = df_clean['PSA_ID'].duplicated(keep=False) & df_clean['PSA_ID'].notna()
df_clean['id_was_rekeyed'] = dup_mask

counter = {}
new_ids = []
for pid, flagged in zip(df_clean['PSA_ID'], dup_mask):
    if flagged:
        counter[pid] = counter.get(pid, 0) + 1
        new_ids.append(f"{pid}_DUP{counter[pid]}")
    else:
        new_ids.append(pid)
df_clean['PSA_ID'] = new_ids

remaining_dupes = df_clean['PSA_ID'].dropna().duplicated().sum()
print(f"Re-keyed {dup_mask.sum()} rows. Remaining duplicate PSA_IDs (excluding still-missing ones): {remaining_dupes}")

Re-keyed 0 rows. Remaining duplicate PSA_IDs (excluding still-missing ones): 0


In [5]:
df_clean['id_was_missing'] = df_clean['PSA_ID'].isna()

missing_mask = df_clean['PSA_ID'].isna()
df_clean.loc[missing_mask, 'PSA_ID'] = [f"GEN_{i:05d}" for i in range(missing_mask.sum())]
print("Remaining missing PSA_IDs:", df_clean['PSA_ID'].isna().sum())

Remaining missing PSA_IDs: 0


3. Normalize, tokenize, flag code-switching

In [6]:
import regex as re2  # handles unicode word boundaries better than re

def normalize_text(text: str) -> str:
    if pd.isna(text):
        return text
    text = str(text).strip()
    text = re2.sub(r'\s+', ' ', text)
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    return text

for col in ['English', 'Kiswahili', 'Dholuo']:
    df_clean[col] = df_clean[col].apply(normalize_text)

df_clean[['English', 'Kiswahili', 'Dholuo']].head(3)

,English,Kiswahili,Dholuo
0,"PRESS RELEASE: JUNE 29, 2020 The EU through it...","TAARIFA KWA VYOMBO VYA HABARI: JUNI 29, 2020 E...","WECHE MA MUMA: Jun 29, 2020 EU kokalo kuom mig..."
1,This grant will be used by the WHO to support ...,Ruzuku hii itatumiwa na WHO kuunga mkono juhud...,Riwruok mar WHO biro tiyo gi konyno e siro kin...
2,"Specifically, WHO Kenya will boost the respons...","Hasa, WHO Kenya itaongeza juhudi za kukabilian...","E yo makende, WHO Kenya biro jiwo kinda mar dw..."


In [7]:
def tokenize(text: str):
    if pd.isna(text) or text == '':
        return []
    return re2.findall(r"[\w'-]+|[^\w\s]", text, flags=re2.UNICODE)

df_clean['english_tokens'] = df_clean['English'].apply(tokenize)
df_clean['kiswahili_tokens'] = df_clean['Kiswahili'].apply(tokenize)
df_clean['dholuo_tokens'] = df_clean['Dholuo'].apply(tokenize)

df_clean[['English', 'english_tokens']].head(2)

,English,english_tokens
0,"PRESS RELEASE: JUNE 29, 2020 The EU through it...","[PRESS, RELEASE, :, JUNE, 29, ,, 2020, The, EU..."
1,This grant will be used by the WHO to support ...,"[This, grant, will, be, used, by, the, WHO, to..."


In [8]:
ENGLISH_STOPWORDS_ISH = {'the','and','of','to','a','in','for','is','on','with'}  # rough heuristic

def flag_code_switch(text: str) -> bool:
    if pd.isna(text) or text == '':
        return False
    tokens = tokenize(text)
    has_acronym = any(re2.fullmatch(r'[A-Z]{2,}', t) for t in tokens)
    has_english_word = any(t.lower() in ENGLISH_STOPWORDS_ISH for t in tokens)
    return has_acronym or has_english_word

df_clean['kiswahili_code_switch'] = df_clean['Kiswahili'].apply(flag_code_switch)
df_clean['dholuo_code_switch'] = df_clean['Dholuo'].apply(flag_code_switch)

print("Kiswahili sentences flagged for code-switching:", df_clean['kiswahili_code_switch'].sum())
print("Dholuo sentences flagged for code-switching:", df_clean['dholuo_code_switch'].sum())

Kiswahili sentences flagged for code-switching: 1903
Dholuo sentences flagged for code-switching: 3065


In [9]:
cultural_terms_glossary = {
    "SHA": "Social Health Authority (Kenya)",
    "IEBC": "Independent Electoral and Boundaries Commission",
    "WHO": "World Health Organization",
    "ECHO": "EU Civil Protection and Humanitarian Aid Operations",
    "GIZ": "German development agency",
    "county": "devolved administrative unit in Kenya",
}
cultural_terms_glossary

{'SHA': 'Social Health Authority (Kenya)',
 'IEBC': 'Independent Electoral and Boundaries Commission',
 'WHO': 'World Health Organization',
 'ECHO': 'EU Civil Protection and Humanitarian Aid Operations',
 'GIZ': 'German development agency',
 'county': 'devolved administrative unit in Kenya'}

In [10]:
df_clean['has_dholuo'] = df_clean['Dholuo'].notna() & (df_clean['Dholuo'].astype(str).str.strip() != '')
print("Rows with Dholuo:", df_clean['has_dholuo'].sum(), "/", len(df_clean))

Rows with Dholuo: 9539 / 9539


4. Sanity checks and save

In [ ]:
# Full version with tokens + flags
df_clean.to_csv("PSA_Preprocessed_NLLB_Final.csv", index=False)

# Lightweight text-only version
text_only = df_clean[['PSA_ID', 'Domain', 'English', 'Kiswahili', 'Dholuo',
                        'has_template_gap', 'id_was_missing',
                        'kiswahili_code_switch', 'dholuo_code_switch']]
text_only.to_csv("PSA_Preprocessed_NLLB_Light.csv", index=False)
print("Final shape:", df_clean.shape)

Final shape: (9539, 14)
